In [2]:

import re

def parse_atom(s):
    p, a = s.strip().split("(")
    return p.strip(), [x.strip() for x in a.rstrip(")").split(",")]

def split_items(s, sep=","):
    result, depth, start = [], 0, 0
    for i, c in enumerate(s):
        if c == "(":
            depth += 1
        elif c == ")":
            depth -= 1
        elif c == sep and depth == 0:
            result.append(s[start:i].strip())
            start = i + 1
    result.append(s[start:].strip())
    return result

def match(pattern, fact, sub):
    if pattern[0] != fact[0]:
        return None
    sub = sub.copy()
    for x, y in zip(pattern[1], fact[1]):
        if x.islower():
            if x in sub and sub[x] != y:
                return None
            sub[x] = y
        elif x != y:
            return None
    return sub

def forward_chain(facts, rules, goal):
    changed = True

    while changed:
        changed = False

        for premises, conclusion in rules:
            def solve(i, sub):
                if i == len(premises):
                    return sub

                for fact in facts:
                    new_sub = match(premises[i], fact, sub)
                    if new_sub is not None:
                        result = solve(i + 1, new_sub)
                        if result is not None:
                            return result
                return None

            sub = solve(0, {})

            if sub is not None:
                new_fact = (
                    conclusion[0],
                    [sub.get(x, x) for x in conclusion[1]]
                )

                if new_fact not in facts:
                    facts.append(new_fact)
                    changed = True

    print("\nDerived Facts:")
    for f in facts:
        print(f"{f[0]}({','.join(f[1])})")

    return goal in facts


print("=== INTERACTIVE FOL FORWARD CHAINING ===")

n = input("\nEnter facts separated by commas:\n")
facts = [parse_atom(x) for x in split_items(n)]

rules = []

print("\nEnter rules one by one (blank to stop):")

while True:
    r = input("Rule: ").strip()
    if not r:
        break

    left, right = r.split("=>")
    premises = [parse_atom(x) for x in left.split("&")]
    conclusion = parse_atom(right)
    rules.append((premises, conclusion))

goal = parse_atom(input("\nEnter the goal to prove:\n"))

print("\n=== RUNNING FORWARD CHAINING ===")
print("Initial Facts:")
for f in facts:
    print("-", f"{f[0]}({','.join(f[1])})")

print("\nRules:")
for p, c in rules:
    left = " & ".join(
        f"{x[0]}({','.join(x[1])})" for x in p
    )
    right = f"{c[0]}({','.join(c[1])})"
    print("-", left, "=>", right)

if forward_chain(facts, rules, goal):
    print("\nSuccess! Goal has been proved.")
    print("True")
else:
    print("\nGoal cannot be proved.")
    print("False")


=== INTERACTIVE FOL FORWARD CHAINING ===

Enter facts separated by commas:
American(Robert), Enemy(CountryA,America), Owns(CountryA,Missiles), Missile(Missiles)

Enter rules one by one (blank to stop):
Rule: American(x) & Weapon(y) & Sells(x,y,z) & Hostile(z) => Criminal(x)
Rule: Enemy(z,America) => Hostile(z)
Rule: Missile(x) => Weapon(x)
Rule: Missile(y) & Owns(z,y) => Sells(Robert,y,z)
Rule: 

Enter the goal to prove:
Criminal(Robert)

=== RUNNING FORWARD CHAINING ===
Initial Facts:
- American(Robert)
- Enemy(CountryA,America)
- Owns(CountryA,Missiles)
- Missile(Missiles)

Rules:
- American(x) & Weapon(y) & Sells(x,y,z) & Hostile(z) => Criminal(x)
- Enemy(z,America) => Hostile(z)
- Missile(x) => Weapon(x)
- Missile(y) & Owns(z,y) => Sells(Robert,y,z)

Derived Facts:
American(Robert)
Enemy(CountryA,America)
Owns(CountryA,Missiles)
Missile(Missiles)
Hostile(CountryA)
Weapon(Missiles)
Sells(Robert,Missiles,CountryA)
Criminal(Robert)

Success! Goal has been proved.
True
